<a href="https://colab.research.google.com/github/sriharikrishna/JDEV-Tutorial/blob/python/04MIMO/04MIMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Multiple Inputs, Multiple Outputs (MIMO)

This example computes affinity and similarity matrices from point coordinates and parameters. The function maps multiple inputs to multiple outputs.

In [ ]:
import jax
import jax.numpy as jnp

def norm_pts_jax(pt1_coords, pt2_coords):
    """Calculates the squared Euclidean distance between two points."""
    return jnp.sum((pt1_coords - pt2_coords)**2, axis=-1)

In [ ]:
def compute_matrices(nb_pts, sigma, max_val, pts_coords, connections_indices):
    """
    Computes affinity and similarity matrices, and updates point affinities.

    Returns:
        total_affinity: scalar sum of squared final affinities
        A: (nb_pts, nb_pts) normalized distance matrix
        S: (nb_pts, nb_pts) similarity matrix
    """
    diff = pts_coords[:, None, :] - pts_coords[None, :, :]
    dist_sq_matrix = jnp.sum(diff**2, axis=-1)

    A = dist_sq_matrix / (max_val * max_val)
    S = jnp.exp(-A / sigma)

    initial_affinities = jnp.ones(nb_pts, dtype=jnp.float32)

    def update_for_connection(carry, connection_pair):
        current_affinities, current_nb_neighbours = carry
        idx1, idx2 = connection_pair
        similarity_val = S[idx1, idx2]

        current_affinities = current_affinities.at[idx1].multiply(similarity_val)
        current_affinities = current_affinities.at[idx2].multiply(similarity_val)
        current_nb_neighbours = current_nb_neighbours.at[idx1].add(1)
        current_nb_neighbours = current_nb_neighbours.at[idx2].add(1)

        return (current_affinities, current_nb_neighbours), None

    (final_affinities, _), _ = jax.lax.scan(
        update_for_connection,
        (initial_affinities, jnp.zeros(nb_pts, dtype=jnp.int32)),
        connections_indices
    )

    total_affinity = jnp.sum(final_affinities**2)
    return total_affinity, A, S

In [ ]:
nb_pts = 5
sigma = 20.0
max_val = 10.0

pts_coords = jnp.array([
    [1.0, 1.0, 1.0],
    [0.0, 1.0, 1.0],
    [0.0, 0.0, 1.0],
    [-1.0, -1.0, 1.0],
    [-1.0, -1.0, -1.0]
], dtype=jnp.float32)

connections = []
for i in range(nb_pts):
    for j in range(i + 1, nb_pts, i + 1 if i > 0 else 1):
        connections.append([i, j])
connections_indices = jnp.array(connections, dtype=jnp.int32)

print(f"sigma = {sigma} \t -- \t max_val = {max_val}")
print("Connections:")
for conn_idx in connections_indices:
    print(f"  ({conn_idx[0]}, {conn_idx[1]})")

total_affinity, A, S = compute_matrices(
    nb_pts, sigma, max_val, pts_coords, connections_indices
)
print(f"Total affinity = {total_affinity}")

### Forward-mode derivatives with `jax.jvp`

1. [`jax.jvp`](https://jax.readthedocs.io/en/latest/jax.html#jax.jvp)
2. Computes $J \cdot v$ for a function $R^m \rightarrow R^n$.
3. `jax.jvp()` requires the input value for the primal function, which is the point where the derivatives are computed.
4. `jax.jvp()` requires a seed $v$. (For forward mode the shape of the seed must match the primal input.)
5. The code below obtains derivatives by calling `jax.jvp()` with Cartesian basis vectors as seeds.

`compute_matrices` returns three outputs `(total_affinity, A, S)`, so one `jax.jvp` call returns three tangents. To get $\partial/\partial \sigma$ or $\partial/\partial \text{max\_val}$, use tangent seeds `(1.0, 0.0)` or `(0.0, 1.0)` on those scalar arguments.

In [ ]:
def compute_from_params(sigma, max_val):
    return compute_matrices(nb_pts, sigma, max_val, pts_coords, connections_indices)

primals, tangents_dsigma = jax.jvp(compute_from_params, (sigma, max_val), (1.0, 0.0))
_, tangents_dmax = jax.jvp(compute_from_params, (sigma, max_val), (0.0, 1.0))

aff, A_primal, S_primal = primals
daff_dsigma, dA_dsigma, dS_dsigma = tangents_dsigma
daff_dmax, dA_dmax, dS_dmax = tangents_dmax

print(f"Primal total affinity = {aff}")
print(f"d(total_affinity)/d(sigma) = {daff_dsigma}")
print(f"d(total_affinity)/d(max_val) = {daff_dmax}")
print(f"d(A)/d(sigma) shape = {dA_dsigma.shape}, d(S)/d(sigma) shape = {dS_dsigma.shape}")

For a vector input such as `pts_coords`, the tangent must have the same shape. Setting one entry of the tangent to `1.0` (and the rest to `0.0`) differentiates with respect to that coordinate while producing tangents for every output.

In [ ]:
def compute_from_coords(coords):
    return compute_matrices(nb_pts, sigma, max_val, coords, connections_indices)

tangent_coords = jnp.zeros_like(pts_coords).at[0, 0].set(1.0)

primals, tangents = jax.jvp(compute_from_coords, (pts_coords,), (tangent_coords,))
daff_dx00, dA_dx00, dS_dx00 = tangents

print(f"d(total_affinity)/d(pts_coords[0,0]) = {daff_dx00}")
print(f"d(A)/d(pts_coords[0,0]) at A[0,1] = {dA_dx00[0, 1]}")
print(f"d(S)/d(pts_coords[0,0]) at S[0,1] = {dS_dx00[0, 1]}")